In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

imagen_bgr = cv2.imread('an3.jpeg')
imagen_rgb = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2RGB)

gris = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2GRAY)
gris = cv2.GaussianBlur(gris, (5, 5), 1)
gris = cv2.equalizeHist(gris)

bordes = cv2.Canny(gris, 50, 150)

lineas = cv2.HoughLinesP(bordes, 1, np.pi/180, threshold=80, minLineLength=50, maxLineGap=10)

centro_x = imagen_bgr.shape[1] // 2
centro_y = imagen_bgr.shape[0] // 2

def calcular_angulo(x1, y1, x2, y2):
    dist1 = np.hypot(x1 - centro_x, y1 - centro_y)
    dist2 = np.hypot(x2 - centro_x, y2 - centro_y)
    if dist1 > dist2:
        dx = x1 - centro_x
        dy = centro_y - y1
    else:
        dx = x2 - centro_x
        dy = centro_y - y2
    angulo = np.degrees(np.arctan2(dy, dx))
    angulo = (90 - angulo) % 360    
    return angulo

def numero_a_letras(n):
    unidades = [
        "cero", "uno", "dos", "tres", "cuatro", "cinco",
        "seis", "siete", "ocho", "nueve", "diez", "once",
        "doce", "trece", "catorce", "quince", "dieciséis",
        "diecisiete", "dieciocho", "diecinueve", "veinte"
    ]
    decenas = [
        "", "", "veinte", "treinta", "cuarenta", "cincuenta"
    ]

    if n <= 20:
        return unidades[n]
    else:
        d = n // 10
        u = n % 10
        if u == 0:
            return decenas[d]
        else:
            return f"{decenas[d]} y {unidades[u]}"


manecillas = []
if lineas is not None:
    for linea in lineas:
        x1, y1, x2, y2 = linea[0]
        longitud = np.hypot(x2 - x1, y2 - y1)
        distancia_centro = min(np.hypot(x1 - centro_x, y1 - centro_y), np.hypot(x2 - centro_x, y2 - centro_y))
        if distancia_centro < 100 and 40 < longitud < 400:  # Aumentamos longitud máxima
            angulo = calcular_angulo(x1, y1, x2, y2)
            manecillas.append({
                'puntos': (x1, y1, x2, y2),
                'longitud': longitud,
                'angulo': angulo
            })

# Asegurarse de que tenemos al menos 3 manecillas detectadas
if len(manecillas) >= 3:
    # Ordenar por longitud de menor a mayor
    manecillas.sort(key=lambda x: x['longitud'])

    horario = manecillas[0]       # más corta
    minutero = manecillas[1]      # intermedia
    segundero = manecillas[-1]    # más larga

    hora = int((horario['angulo'] / 30) % 12)
    minutos = int((minutero['angulo'] / 6) % 60)
    segundos = int((segundero['angulo'] / 6) % 60)

    # Corrección de hora si está a medio camino
    fraccion_hora = (horario['angulo'] / 30) % 12
    if abs(fraccion_hora - hora) > 0.5:
        hora = (hora + 1) % 12

    # Dibujar las manecillas
    imagen_resultado = imagen_rgb.copy()
    cv2.line(imagen_resultado, (horario['puntos'][0], horario['puntos'][1]), (horario['puntos'][2], horario['puntos'][3]), (255, 0, 0), 8)  # horario - azul grueso
    cv2.line(imagen_resultado, (minutero['puntos'][0], minutero['puntos'][1]), (minutero['puntos'][2], minutero['puntos'][3]), (0, 255, 0), 6)  # minutero - verde
    cv2.line(imagen_resultado, (segundero['puntos'][0], segundero['puntos'][1]), (segundero['puntos'][2], segundero['puntos'][3]), (0, 0, 255), 2)  # segundero - rojo fino

    plt.figure(figsize=(10, 10))
    plt.imshow(imagen_resultado)
    texto_hora = f"{numero_a_letras(hora)} con {numero_a_letras(minutos)} y {numero_a_letras(segundos)} segundos"
    plt.title(f'Hora: {texto_hora}')
    plt.axis('off')
    plt.show()
